# Process an EIGER Scan With Saved Detector Corrections

## Purpose

Apply the detector-pixel and eight-sensor response learned by `extract_eiger_corrections.ipynb` to a compatible raw scan. This notebook **does not refit detector gains** and does not need the extraction notebook's kernel. It loads the saved calibration, uses the target scan's own measured I0 and native mask, integrates frames on common q and 2theta grids, and saves a processed-result HDF5 plus a plain-text 2theta `.xy` file.

## How To Use

1. Keep `eiger_workflow.py` beside the notebooks. Select a kernel containing NumPy, h5py, matplotlib, pyFAI, and xraylib; run the interpreter check.
2. Set `SCAN_PATH` to the raw target NeXus file and `CALIBRATION_PATH` to an output of the extraction notebook. The initial example processes a compatible scan using detector gains extracted from 99877.
3. Choose a new `RESULT_OUTPUT`; its matching `.xy` output is derived automatically. Existing HDF5 or `.xy` files are never overwritten. Companion detector, monitor, and metadata filenames are inferred from the scan stem unless explicitly overridden with the corresponding path settings.
4. Set `Q_RANGE_NM`, `Q_BIN_COUNT`, and `TWO_THETA_BIN_COUNT`. The q result uses the requested interval; the 2theta result spans full detector coverage over all scan positions. `ALLOW_OUTSIDE_FIT_RANGE=True` explicitly accepts detector-response extrapolation in the full 2theta result. The saved `within_detector_fit_q_range` flags identify that extrapolation.
5. Keep `APPLY_SOURCE_SCAN_FRAME_GAINS=False` for subsequent scans. Setting it True is permitted only for the exact calibration source scan with identical recorded angles and I0. It never estimates new frame gains or applies another scan's frame gains.
6. Run compatibility/loading, processing/plots, HDF5 saving, then the final `.xy` export cell. `WRITE_RESULT=False` allows inspection without writing either result. Once outputs have been written, choose a new output name before rerunning the settings/export cells.

## Compatibility and Corrections

The supported detector is the documented EIGER2 CdTe 500K, shape 512 x 1028. The loader verifies frame/angle/I0 lengths, positive finite monitors, and detector metadata. Energy, wavelength, sensor thickness/material, orientation, and the saved geometry convention must match the calibration. A mismatch stops processing; using a calibration at another energy or geometry requires deliberate validation and potentially a new extraction, not silently bypassing these checks.

The geometry is read from the calibration's embedded JSON. Polarization, absolute solid angle, and modeled CdTe detector absorption are applied using its saved settings, with the same convention as extraction. The absorbed-fraction model is based on xraylib total attenuation and assumed density, not measured photon-counting QE. A geometry check confirms vertical lab-frame rotation for the horizontal-polarization convention.

Raw detector counts are divided once by the saved combined pixel/sensor divisor. Poisson variance starts from raw counts and is divided by the divisor squared. Current native-mask pixels are combined with saved native/boundary exclusions and the saved persistent pixel candidates. Nonfinite, negative, and high/fill-sentinel counts are excluded dynamically using `SATURATION_THRESHOLD`.

There is no reference-based rejection against the calibration scan's scattering pattern: later samples need not have the same scattering. There is no fitted normalization for a new target scan, no tapering, and no final-pattern smoothing. The producer's per-frame fitting rejections and calibration-scan frame gains are not automatically carried into unrelated scans. Even for source-scan processing, model-based per-contribution rejection from the research notebook is not replayed; this application workflow uses saved permanent masks and raw-value dynamic masks.

## Integration and Merging

The CPU bbox/CSR method fractionally splits pixels across bins. All frames use identical radial bins for each output. For each bin the merged intensity is total `sum_signal` / total `sum_normalization`; counting sigma is sqrt(total `sum_variance`) / total `sum_normalization`. Normalization includes measured I0, absolute solid-angle prefactor, relative solid angle, polarization, and detector absorption. Per-frame accumulators are retained and their sum is checked.

A frame can contribute no pixels to a q range while still contributing to the full 2theta result; empty-CSR warnings for such q integrations do not represent measured zero intensity. Only bins with positive normalization are plotted. Gain maps are verified unchanged during processing, and the notebook explicitly tests energy-mismatch and frame-gain-transfer safeguards.

## Processed HDF5 and XY Layout

The HDF5 schema identifier is `eiger_processed_v1`:

- `entry/merged_q/`: `q_nm`, intensity, sigma, signal/normalization/variance/count accumulators, per-frame accumulators, fit-range flags, angles, measured I0, applied frame gains, and effective I0.
- `entry/merged_2theta/`: the same statistics on `two_theta_deg` over full scan coverage.
- `entry/calibration/`: exact applied gain maps, geometry JSON, permanent/fixed mask, and calibration-source provenance.
- `entry/physical_corrections/`: the saved physical-model parameters, with absolute solid-angle normalization enabled.
- Root attributes: source scan, calibration path, schema, no-refinement flag, and uncertainty limitations.

The matching `.xy` file contains two ASCII columns for supported 2theta bins only: `2theta_deg intensity`. It uses the same merged, physically corrected intensity as `entry/merged_2theta/intensity`; unsupported bins are omitted rather than written as zeros. The save routines create new files exclusively. The default tested HDF5 output is `i15-1-99878_processed_with_saved_corrections.h5`, with an `.xy` file of the same basename.

## Interpretation and Troubleshooting

The shaded plot interval marks the q range used to fit the detector correction. Outside it the map is an extrapolation, not independently validated response. Very strong low-angle sample scattering can dominate the full-range plot and does not imply calibration failure.

Sigma is propagated counting uncertainty only; it excludes calibration, I0, attenuation/polarization-model uncertainty and covariance between neighbouring split bins. More precise statistical errors do not guarantee absence of systematic structure. The calibration is exploratory until independently validated.

A missing calibration must first be produced by notebook 1; earlier research exports have a different schema and are rejected. For missing companion files, set their explicit path overrides. The optional OpenCL warning is harmless for the CPU method. Do not supply already corrected images, use source-scan frame gains on a different scan, or overwrite historical results. The original exploratory notebook remains unchanged.

In [ ]:
import sys
print('Python executable:', sys.executable)
print('Python version:', sys.version)

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import h5py
import matplotlib.pyplot as plt

REPO_DIRECTORY = Path.cwd().resolve()
while REPO_DIRECTORY != REPO_DIRECTORY.parent and not (REPO_DIRECTORY / 'eiger_workflow.py').exists():
    REPO_DIRECTORY = REPO_DIRECTORY.parent
if not (REPO_DIRECTORY / 'eiger_workflow.py').exists():
    raise FileNotFoundError('Could not find eiger_workflow.py; launch Jupyter from the repository or a subdirectory.')
DATA_DIRECTORY = REPO_DIRECTORY / 'data'
DATA_DIRECTORY = Path("/dls/i15-1/data/2026/cm44163-4")
OUTPUT_DIRECTORY = REPO_DIRECTORY / 'outputs'
OUTPUT_DIRECTORY.mkdir(exist_ok=True)

SCAN_PATH = DATA_DIRECTORY / 'i15-1-100603.nxs'
CALIBRATION_PATH = OUTPUT_DIRECTORY / 'i15-1-100603_detector_corrections.h5'







RESULT_OUTPUT = OUTPUT_DIRECTORY / f'{SCAN_PATH.stem}_processed_with_saved_corrections.h5'
TWO_THETA_XY_OUTPUT = RESULT_OUTPUT.with_suffix('.xy')
Q_RANGE_NM = (200.0, 400.0)
Q_BIN_COUNT = 800
TWO_THETA_BIN_COUNT = 1000
APPLY_SOURCE_SCAN_FRAME_GAINS = False
ALLOW_OUTSIDE_FIT_RANGE = True
SATURATION_THRESHOLD = 1e7
WRITE_RESULT = True
DETECTOR_PATH = None
MONITOR_PATH = None
METADATA_PATH = None

assert SCAN_PATH.exists() and CALIBRATION_PATH.exists()
assert not RESULT_OUTPUT.exists(), 'Choose a new RESULT_OUTPUT; existing files are never overwritten.'
assert not TWO_THETA_XY_OUTPUT.exists(), 'Choose a new output name; existing XY files are never overwritten.'
if str(REPO_DIRECTORY.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIRECTORY.resolve()))
import eiger_workflow as workflow
importlib.reload(workflow)
processing_scan = workflow.load_scan(SCAN_PATH, DETECTOR_PATH, MONITOR_PATH, METADATA_PATH)
saved_calibration = workflow.load_calibration(CALIBRATION_PATH)
processing_model = workflow.model_for_processing(processing_scan, saved_calibration)
print('Repository:', REPO_DIRECTORY.resolve())
print('Raw scan:', processing_scan.path)
print('Detector calibration learned from:', saved_calibration['source_scan'])
print('Detector shape / energy / material:', processing_scan.shape, processing_scan.energy_kev, processing_scan.material)
print('Geometry and physical settings loaded from the calibration file, not from a previous notebook kernel.')
print('Source-scan frame gains enabled:', APPLY_SOURCE_SCAN_FRAME_GAINS)
print('New result outputs:', RESULT_OUTPUT.resolve(), TWO_THETA_XY_OUTPUT.resolve())

In [ ]:
from dataclasses import replace

try:
    workflow.model_for_processing(replace(processing_scan, energy_kev=processing_scan.energy_kev * 1.1), saved_calibration)
except ValueError as error:
    print('Energy-mismatch safeguard verified:', error)
else:
    raise AssertionError('Energy mismatch was not rejected')
if str(processing_scan.path) != saved_calibration['source_scan']:
    try:
        workflow.process_scan(processing_scan, saved_calibration, processing_model, apply_calibration_frame_gains=True)
    except ValueError as error:
        print('Frame-gain transfer safeguard verified:', error)
    else:
        raise AssertionError('Calibration-frame gains were transferred to another scan')

original_detector_divisor = saved_calibration['combined_divisor'].copy()
processed = workflow.process_scan(
    processing_scan, saved_calibration, processing_model,
    q_range=Q_RANGE_NM, q_npt=Q_BIN_COUNT, theta_npt=TWO_THETA_BIN_COUNT,
    apply_calibration_frame_gains=APPLY_SOURCE_SCAN_FRAME_GAINS,
    allow_extrapolation=ALLOW_OUTSIDE_FIT_RANGE, saturation=SATURATION_THRESHOLD,
)
np.testing.assert_array_equal(saved_calibration['combined_divisor'], original_detector_divisor)
if not APPLY_SOURCE_SCAN_FRAME_GAINS:
    np.testing.assert_array_equal(processed['frame_gains'], np.ones(len(processing_scan.angles)))
print('Saved detector gains unchanged; no fitting performed.')
print('Processed frames:', len(processing_scan.angles))
print('Detector-corrected q overlap:', workflow.overlap_summary(processed['results']['merged_q']))

fig, axes = plt.subplots(2, 1, figsize=(13, 9), constrained_layout=True)
for plot_axis, group_name, xlabel in ((axes[0], 'merged_q', 'q (nm$^{-1}$)'), (axes[1], 'merged_2theta', '2theta (degrees)')):
    group = processed['groups'][group_name]
    valid = group['accumulators'][1] > 0
    plot_axis.plot(group['axis'][valid], group['intensity'][valid], linewidth=0.9, color='tab:blue')
    plot_axis.fill_between(group['axis'][valid], group['intensity'][valid] - group['sigma'][valid], group['intensity'][valid] + group['sigma'][valid], color='tab:blue', alpha=0.2, linewidth=0)
    plot_axis.set(xlabel=xlabel, ylabel='I0 and physically corrected intensity', title=f'{SCAN_PATH.stem}: unchanged saved detector corrections')
    within = group['within_fit_range'] & valid
    if within.any():
        plot_axis.axvspan(group['axis'][within].min(), group['axis'][within].max(), color='black', alpha=0.06, label='Detector-correction fitting range')
        plot_axis.legend(fontsize=8)
    print(group_name, 'axis range:', group['axis'].min(), group['axis'].max(), 'supported bins:', valid.sum(), 'outside fit range:', np.count_nonzero(valid & ~group['within_fit_range']))
plt.show()

frame_numbers = np.arange(1, len(processing_scan.i0) + 1)
fig_i0, axis_i0 = plt.subplots(figsize=(10, 4), constrained_layout=True)
axis_i0.plot(frame_numbers, processing_scan.i0, marker='o', markersize=3, linewidth=1, color='tab:orange')
axis_i0.set(xlabel='Frame number', ylabel='I0', title=f'{SCAN_PATH.stem}: measured I0 by frame')
axis_i0.grid(True, alpha=0.25)
plt.show()
print('Full 2theta coverage outside the fit range is extrapolation, not independently validated response.')

In [ ]:
processed_file_written = None
if WRITE_RESULT:
    processed_file_written = workflow.save_processed(
        RESULT_OUTPUT, processing_scan, saved_calibration, processing_model, processed,
    )
    with h5py.File(processed_file_written, 'r') as result_file:
        np.testing.assert_array_equal(result_file['entry/calibration/combined_divisor'][()], saved_calibration['combined_divisor'])
        if not APPLY_SOURCE_SCAN_FRAME_GAINS:
            np.testing.assert_array_equal(result_file['entry/merged_q/frame_intensity_gains'][()], np.ones(len(processing_scan.angles)))
        print('Created and read-back verified:', processed_file_written)
        print('Result groups:', list(result_file['entry'].keys()))
        print('Detector calibration source:', result_file['entry/calibration'].attrs['source_calibration_scan'])
    print('No detector gains were fitted to this scan. Earlier processed files remain unchanged.')
else:
    print('Export disabled. Set WRITE_RESULT=True when ready to save a new processed HDF5 file.')

In [ ]:
if WRITE_RESULT:
    if processed_file_written is None:
        raise RuntimeError('Run the HDF5 export cell before writing the matching XY file.')
    theta_group = processed['groups']['merged_2theta']
    supported = (theta_group['accumulators'][1] > 0) & np.isfinite(theta_group['axis']) & np.isfinite(theta_group['intensity'])
    xy_data = np.column_stack((theta_group['axis'][supported], theta_group['intensity'][supported]))
    assert xy_data.size > 0, 'No supported 2theta bins are available to export.'
    TWO_THETA_XY_OUTPUT = processed_file_written.with_suffix('.xy')
    with TWO_THETA_XY_OUTPUT.open('x', encoding='ascii') as xy_file:
        np.savetxt(xy_file, xy_data, fmt='%.10g', header='2theta_deg intensity')
    print('Created plain-text 2theta result:', TWO_THETA_XY_OUTPUT.resolve())
    print('Exported supported bins:', len(xy_data))
else:
    print('XY export skipped because WRITE_RESULT=False.')